# The Code-Mixed Pedagogical Flow Extractor

This Python Notebook contains my implementation of [task.ipynb](task.ipynb).  


## Video Selection

I have selected the 5 videos listed below. Most of them are a blend of Kannada and English (Kanglish). This is because Kannada is my mother tongue, and I can speak it fluently.

**Note:** You can run the code either for the 5 example videos, or for your own video. If you want to run it for the example videos, then press the video number.

If you want to run it for your own video, then press `0` when prompted and then paste the YouTube video link to the video you want to run the pipeline all.

The videos we selected are as follows:
1. [Python in Kannada - Introduction to Coding and Python | Full Course for Beginners - #1](https://www.youtube.com/watch?v=8c74mXV2lJ0&list=PLlGueSbLhZoBRnTsGiDJeTXuQCALOTN07) | KANNADA+ENGLISH
2. [Chapter1- The Living World ಕನ್ನಡದಲ್ಲಿ | biology | NCERT](https://www.youtube.com/watch?v=lse_2joLgLo) | KANNADA+ENGLISH
3. [Embryo Sac Development|Sexual Reproduction in flowering plants in kannada| BIOLOGY|NEET](https://www.youtube.com/watch?v=EibWp3ssXi4) | KANNADA+ENGLISH
4. [Banking of Road with friction for JEE & NEET | Class 11 Physics in Minutes](https://www.youtube.com/watch?v=_qrM61-9td8) | HINDI+ENGLISH
5. 

In [1]:
%pip install yt_dlp
%pip install python-dotenv
%pip install google-generativeai
%pip install aksharamukha

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached fonttools-4.61.1-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (114 kB)
Using cached fonttools-4.61.1-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl (5.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.2 MB/s  0:00:00 eta 0:00:01
Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (807 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 12.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 12.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import yt_dlp
import os

# These are the 5 example video links. 
example_videos = {
    "1": "https://www.youtube.com/watch?v=8c74mXV2lJ0",
    "2": "https://www.youtube.com/watch?v=9AaG24KcVIo",
    "3": "https://www.youtube.com/watch?v=uMzUB89uSxU",
    "4": "https://www.youtube.com/watch?v=CJ_VhH6nhko",
    "5": "https://www.youtube.com/watch?v=PLACEHOLDER_LINK_5",
}

print("Select a video to process:")
print("[0] Enter your own custom YouTube link")
for key in example_videos.keys():
    print(f"[{key}] Run example video {key}")

choice = input("Enter your choice (0-5): ")

if choice == "0":
    video_url = input("Paste your YouTube link here: ")
elif choice in example_videos:
    video_url = example_videos[choice]
    print(f"\nSelected Example {choice}: {video_url}")
else:
    print("\nInvalid choice. Defaulting to Example 1.")
    video_url = example_videos["1"]

ydl_opts = {
    'format': 'm4a/bestaudio/best', 
    'outtmpl': 'lesson_audio.%(ext)s', # This will likely save as lesson_audio.m4a
}
import glob
for f in glob.glob("lesson_audio.*"): os.remove(f)

print(f"\nDownloading audio from: {video_url} ...")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    downloaded_file = glob.glob("lesson_audio.*")[0]
    print(f"\nAudio downloaded successfully as {downloaded_file}!")
except Exception as e:
    print(f"\nError downloading video: {e}")

Select a video to process:
[0] Enter your own custom YouTube link
[1] Run example video 1
[2] Run example video 2
[3] Run example video 3
[4] Run example video 4
[5] Run example video 5

Selected Example 1: https://www.youtube.com/watch?v=8c74mXV2lJ0

[youtube] Extracting URL: https://www.youtube.com/watch?v=8c74mXV2lJ0
[youtube] 8c74mXV2lJ0: Downloading webpage


[youtube] 8c74mXV2lJ0: Downloading android vr player API JSON
[info] 8c74mXV2lJ0: Downloading 1 format(s): 140
[download] Destination: lesson_audio.m4a
[download] 100% of   28.38MiB in 00:00:02 at 10.12MiB/s    
[FixupM4a] Correcting container of "lesson_audio.m4a"

Audio downloaded successfully as lesson_audio.m4a!


## Transcription

My first attempt at audio transcription was using OpenAI Whisper, the failed attempt can be found by [clicking here.](code.ipynb#asr---failed-approach). I have also done a robust analysis of why I think this failed, which can be found by [clicking here.]

I instead decided to use Google Gemini, which is a multimodal model that can understand and generate audio, video, and text. This seemed to be the best option, and using the Gemini Pro API was pretty easy to set up.

Gemini works by:
1. No intermittent STT step. The model directly ingests the audio processes raw audio natively.
2. The raw audio waveform is chopped into continuous frames and encoded directly into mathematical vectors (tokens). These tokens are projected directly into the model's latent space.
3. These acoustic tokens are interleaved seamlessly with the text tokens from your prompt. The neural network's transformer layers process the audio and the text simultaneously.

I was very satisfied with how Gemini Pro performed. It was able to transcribe the audio verbatim, and the output was very high quality. I was also satisfied with the speed of the transcription, which was very fast.

In [ ]:
import os
import time
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("API Key not found! Please make sure your .env file is set up correctly.")

genai.configure(api_key=API_KEY)

import glob
audio_file_path = glob.glob("lesson_audio.*")[0]

print(f"Uploading {audio_file_path} to Gemini...")
audio_file = genai.upload_file(path=audio_file_path)
print("Upload complete. Asking Gemini to transcribe...")

model = genai.GenerativeModel('gemini-pro-latest')

prompt = """
Listen to this audio file. The speaker is using heavily code-mixed speech (a native regional language mixed with English). 
Please transcribe the audio verbatim. 

CRITICAL RULES:
1. Auto-detect the primary native language being spoken (e.g., Hindi, Tamil, Kannada, etc.).
2. Do NOT translate the native language into English. 
3. Write the native language portions using their authentic native script (e.g., Devanagari for Hindi, Kannada script for Kannada, etc.).
4. Write the English words, mathematical terms, and numbers strictly in English.
5. Do NOT output anything except the exact verbatim transcription. No introductions, no conversational text, and no markdown blocks.
"""

start_time = time.time()
# Pass both the text prompt and the audio file object to the multimodal model
response = model.generate_content([prompt, audio_file])
end_time = time.time()

print(f"\nTranscription complete in {round(end_time - start_time, 2)} seconds!\n")

transcript = response.text

# Save to file
with open("gemini_transcript.txt", "w", encoding="utf-8") as f:
    f.write(transcript)


Uploading lesson_audio.m4a to Gemini...
Upload complete. Asking Gemini to transcribe...

Transcription complete in 91.93 seconds!



## Transliteration

Now, I'm going to transliterate the raw transcription into Roman text. This makes the data accessible to anyone reading the notebook and gives us a standardized baseline to elaborate on the Hinglish/Kanglish colloquialisms.

Why not just use Gemini for this? Why not just tweak the prompt we just wrote? It comes down to determinism versus probability. LLMs are probabilistic generators, meaning their spelling and phonetic translations will inevitably drift. I need this transliteration to be a rigid, 1-to-1 character mapping. Offloading this to a deterministic, rule-based open-source library ensures that a specific Kannada word is transliterated the exact same way, every single time we run the cell

For the transliteration, we'll use the open-source library, [github.com/virtualvinodh/aksharamukha-python](github.com/virtualvinodh/aksharamukha-python).  

I chose this library since it supports more than 120 scripts and more than 21 Romanisation methods. I believe that this overall pipeline / framework should be built to work irrespective of the language we're working with.

I searched the internet and learned about aa few of the different Romanisation methods. The comparative differences can be found [here](code.ipynb#comparison-between-techniques-of-transliteration), in the appendix. After consideration, I have decided to use ISO 15919 for transliteration.

In [2]:
import os
from aksharamukha import transliterate

input_file = "gemini_transcript.txt"
output_file = "transliterated_transcript.txt"

if not os.path.exists(input_file):
    print(f"Error: '{input_file}' not found. Please ensure the file exists in your directory.")
else:
    print(f"Reading raw code-mixed transcript from '{input_file}'...")
    
    with open(input_file, "r", encoding="utf-8") as f:
        raw_transcript = f.read()
        
    print("Applying deterministic Aksharamukha ISO 15919 transliteration...")
    
    # 'autodetect' automatically finds the Indic script (e.g., Kannada or Devanagari)
    # 'ISO' maps it to the lossless academic standard.
    # English text (Latin script) is natively ignored and left perfectly intact.
    clean_text = transliterate.process("autodetect", "ISO", raw_transcript)
    
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(clean_text)
        
    print(f"Transliteration complete! Saved to '{output_file}'\n")

Reading raw code-mixed transcript from 'gemini_transcript.txt'...
Applying deterministic Aksharamukha ISO 15919 transliteration...
Transliteration complete! Saved to 'transliterated_transcript.txt'



## Concept Extraction

From the transliterated text, we will now identify the core concepts which were taught.  
For this part of the pipeline, I will again use the Gemini Pro API.  

Parsing the transcript through Gemini with a clear, instruction-based prompt will allow it's newer models to extract new concepts.  

**How this happens:**  


---
---

# APPENDIX
## ASR - Failed Approach

Here we do Automatic Speech Recognition (ASR) using [OpenAI Whisper](https://openai.com/index/whisper/).  
Whisper is trained on 680,000 hours of multilingual and multitask supervised data. It shows high robustness to accents, background noise and technical languages.

We will be using it for **transcription only**, and not translation or transliteration. This is because I want to keep the Indic terms and code-mixed analogies constant.  

I use the `medium` version of the model, which is about 769M parameters and requires ~5GB VRAM, since I want it to be robust to multilinguality as well as accents. I am not using `large-v2` and `large-v3`, since these are 1.5B+ parameters, and require about 10GB of VRAM to run comfortably. Additionally, the large models are slower as well. For the usecase of this task, `medium` should do fine.  

We are running the model locally. This is because I hit rate limits sending large, 10-15 minute audio files. Also, it sort of adds reproducibility, since now anyone who clones and can run the code self-contained.

I tried running the base-model for testing, and it spewed gibberish:  
"0,2,4,6,8 nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd nd"

to be exact... It could not at all handle the rapid kannada-english code switching in that video, would be my supposition.

**THIS IS NOT WORKING**  
Even with the `large-v3` model, it spewed gibberish. And this gibberish was spewed after **2 hours** of running on Google Colab. To some extent, it's a model skill issue. I will not be using ASR for transcription. I will do a robust analysis of why it failed as well.

In [ ]:
# This has been INTENTIONALLY commented out so that it doesn't run and throw errors when we do click `run all`

'''
import whisper
import time
import os

import glob
audio_file = glob.glob("lesson_audio.*")[0]

if not os.path.exists(audio_file):
    print(f"Error: {audio_file} not found. Please run the download cell first.")
else:
    model = whisper.load_model("medium") 
    
    print(f"\nTranscribing {audio_file}... this might take a few minutes.")
    start_time = time.time()
    
    # We force the language to Kannada ("kn") to prevent auto-translation to English.
    # The other parameters prevent the model from getting stuck in a repetition loop.
    result = model.transcribe(
        audio_file,
        language="kn", 
        condition_on_previous_text=False, 
        fp16=False, 
        no_speech_threshold=0.6
    )
    
    end_time = time.time()
    print(f"\nTranscription complete in {round(end_time - start_time, 2)} seconds!")
    
    transcript = result["text"]
    
    # Save the transcript to a text file for the next step (LLM processing)
    with open("transcript.txt", "w", encoding="utf-8") as f:
        f.write(transcript)
        
    print("Full transcript successfully saved to 'transcript.txt'.")


SyntaxError: incomplete input (302848402.py, line 1)

## Why OpenAI Whisper Failed

## Comparison between techniques of transliteration

| Romanization Format | Primary Mechanism | Handles Dravidian? | Case-Sensitive? (Will `.lower()` break it?) | Best Use Case / Verdict |
| :--- | :--- | :--- | :--- | :--- |
| **ISO 15919** | Diacritics (`ā`, `ṭ`, `ē`) | **Yes** | **No**  | **Gold Standard.** Lossless, safe for lowercase operations, and natively supports Kannada/Tamil. |
| **IAST** | Diacritics (`ā`, `ṭ`) | No | **No**  | The academic standard for **Sanskrit/Hindi only**. Fails on Dravidian short/long vowels. |
| **Harvard-Kyoto** | Capitalization (`A`, `T`) | No | **Yes**  | Legacy typing. A clean ASCII format for 90s emails, but terrible for modern data pipelines. |
| **ITRANS** | Capitalization & Multi-char (`aa`, `T`) | Messy / Limited | **Yes**  | Legacy internet standard. Great for human typing on QWERTY, terrible for deterministic machine parsing. |
| **Velthuis** | Double chars (`aa`, `.t`) | No | **No**  | An older alternative to HK that avoids case-sensitivity by using punctuation. Obsolete compared to ISO. |
| **WX / SLP1** | 1-to-1 ASCII mapping (`A`, `w`) | No | **Yes**  | Built specifically for backend Python regex and algorithmic parsing in Indian linguistic research. |